# STAT/CS 2870 Final Project
## Getting Started: Loading the Data

This notebook loads the four project datasets, fixes the first problem in each, and takes a first look.

**Before you start**

1. Download the CSV file(s) from Brightspace.
2. Put them in the **same folder** as this notebook. Do not rename them.
3. Run the cells in order, from top to bottom.

Your team only needs **its own dataset**. Each dataset section below works on its own, so you can skip the others. The **Setup** cell must always run first.

## Setup

In [1]:
import os
import pandas as pd

# Show every column when printing a DataFrame
pd.set_option('display.max_columns', None)

# Check which data files are in this folder
files = ['I_Burlington_Trees.csv',
         'Burlington_Police_Department_Arrests.csv',
         'Burlington_Police_Department_Uses_of_Force.csv',
         'Traffic_Stops.csv']
for f in files:
    status = 'found' if os.path.exists(f) else 'not found (only a problem if your team uses this dataset)'
    print(f'{f}: {status}')

I_Burlington_Trees.csv: not found (only a problem if your team uses this dataset)
Burlington_Police_Department_Arrests.csv: found
Burlington_Police_Department_Uses_of_Force.csv: not found (only a problem if your team uses this dataset)
Traffic_Stops.csv: found


---
## Dataset 1: Burlington Trees

**The first fix:** the `Geo Point` column holds latitude and longitude together in one text string, like `"44.448,-73.228"`. We split it at the comma into two numeric columns, `lat` and `lon`.

In [2]:
trees = pd.read_csv('I_Burlington_Trees.csv')
trees[['lat', 'lon']] = trees['Geo Point'].str.split(',', expand=True).astype(float)

print(trees.shape)
trees.head()

FileNotFoundError: [Errno 2] No such file or directory: 'I_Burlington_Trees.csv'

### First look: column types and missing values

In [3]:
trees.info()

NameError: name 'trees' is not defined

### Check: not every row is a tree

Some rows are stumps, removals, or empty planting sites. Decide whether your question should keep them.

In [4]:
trees['site_typ'].value_counts()

NameError: name 'trees' is not defined

### Check: zeros that mean "missing"

In this file, a `0` often means *not measured* rather than a real zero. Count them before you compute averages or fit a model.

In [5]:
(trees[['diameter', 'height', 'spread', 'conditn', 'planted']] == 0).sum()

NameError: name 'trees' is not defined

---
## Dataset 2: Burlington Police Department Arrests

**The first fixes:**

- The file starts with a hidden character (a *byte-order mark*). Reading it with `encoding='utf-8-sig'` removes it. Without this, the first column's name comes out garbled.
- `arrest_date` is stored as text. `pd.to_datetime` turns it into real dates you can sort, filter, and group by year or month.
- `age` is stored as text. `pd.to_numeric(..., errors='coerce')` converts it to numbers and turns anything unreadable into `NaN`.

In [6]:
arrests = pd.read_csv('Burlington_Police_Department_Arrests.csv',
                      encoding='utf-8-sig', low_memory=False)
arrests['arrest_date'] = pd.to_datetime(arrests['arrest_date'], utc=True)
arrests['age'] = pd.to_numeric(arrests['age'], errors='coerce')

print(arrests.shape)
arrests.head()

(25208, 21)


,OBJECTID,incident_number,arrest_date,gender,race,age,charge,most_serious,ethnicity,felony,violent,category,arrest_type,addl_charge_1,addl_charge_2,addl_charge_3,addl_charge_4,addl_charge_5,addl_charge_6,addl_charge_7,addl_charge_8
0,1,12BU000021,2012-01-01 02:23:00+00:00,NaN,NaN,NaN,Simple Assault 13 VSA 1023 13b,1.0,NaN,False,True,Assault,Flash Cite (no prints/photos) - S,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,12BU000029,2012-01-01 04:01:00+00:00,Male - M,White - W,26.0,Disorderly Conduct - All Other 13 VSA 1026 90c,1.0,Not Hispanic or Latino - N,False,False,Disorder,Cited (no prints/photos) - S,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,12BU000031,2012-01-01 04:30:00+00:00,Male - M,White - W,54.0,Domestic Assault 13 VSA 1042 13b,1.0,Not Hispanic or Latino - N,False,True,DV,Lodged - T,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,12BU000050,2012-01-01 12:45:00+00:00,Male - M,White - W,26.0,Grand Larceny 13 VSA 2501 23h,1.0,Not Hispanic or Latino - N,True,False,Property,Lodged - T,Petit Larceny from a Motor Vehicle 13 VSA 2502...,Unlawful Mischief (Misdemeanor) 13 VSA 3701(B)...,NaN,NaN,NaN,NaN,NaN,NaN
4,5,12BU000049,2012-01-01 13:10:00+00:00,NaN,NaN,NaN,Simple Assault 13 VSA 1023 13b,1.0,NaN,False,True,Assault,Cited (no prints/photos) - S,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### First look: column types and missing values

In [7]:
arrests.info()

<class 'pandas.DataFrame'>
RangeIndex: 25208 entries, 0 to 25207
Data columns (total 21 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   OBJECTID         25208 non-null  int64              
 1   incident_number  25208 non-null  str                
 2   arrest_date      25208 non-null  datetime64[us, UTC]
 3   gender           24003 non-null  str                
 4   race             24008 non-null  str                
 5   age              22571 non-null  float64            
 6   charge           23981 non-null  str                
 7   most_serious     25189 non-null  float64            
 8   ethnicity        21497 non-null  str                
 9   felony           20289 non-null  object             
 10  violent          21502 non-null  object             
 11  category         24603 non-null  str                
 12  arrest_type      25208 non-null  str                
 13  addl_charge_1    4792 non-n

### Check: impossible ages

Look at the minimum and maximum. Are they believable for an arrest?

In [ ]:
arrests['age'].describe()

### Check: labels with codes, and "Expunged"

Categories include a code after the label (`White - W`), and `Expunged` appears as a value. Decide how to handle both.

In [ ]:
arrests['race'].value_counts(dropna=False)

### Check: date range

In [ ]:
print('Earliest arrest:', arrests['arrest_date'].min())
print('Latest arrest:  ', arrests['arrest_date'].max())

---
## Dataset 3: Burlington Police Department Uses of Force

**The first fixes:** this file has the same hidden character as the Arrests file, so it also needs `encoding='utf-8-sig'`. `call_time` is text and is converted to dates.

In [ ]:
force = pd.read_csv('Burlington_Police_Department_Uses_of_Force.csv', encoding='utf-8-sig')
force['call_time'] = pd.to_datetime(force['call_time'], utc=True)

print(force.shape)
force.head()

### First look: column types and missing values

In [ ]:
force.info()

### Check: the record system changed over time

The `database_` column shows which system each record came from. A change over time in your results could be a real change, or just a change in how records were kept.

In [ ]:
pd.crosstab(force['year'], force['database_'])

### Check: blanks and the word "MISSING"

Some columns have both empty values (`NaN`) and the text `MISSING`. Both mean the same thing. Also look for labels that are spelled two ways, such as `Verbal Non-Compliance` and `Verbal Noncompliance`.

In [ ]:
force['subject_action'].value_counts(dropna=False)

---
## Dataset 4: Burlington Police Department Traffic Stops

**The first fixes:** this file also starts with the hidden character, so it needs `encoding='utf-8-sig'`. `call_time` is text and is converted to dates.

In [ ]:
stops = pd.read_csv('Traffic_Stops.csv', encoding='utf-8-sig')
stops['call_time'] = pd.to_datetime(stops['call_time'], utc=True)

print(stops.shape)
stops.head()

### First look: column types and missing values

In [ ]:
stops.info()

### Check: stops per year

The number of stops changes a lot over time, and the last year has only one month. Look at this before you compare years.

In [ ]:
stops['call_time'].dt.year.value_counts().sort_index()

### Check: impossible ages

Look at the minimum and maximum. Are they believable for a driver?

In [ ]:
stops['age'].describe()

### Check: codes inside the labels

Outcomes are stored as `code = label`. Notice how uneven the counts are.

In [ ]:
stops['stop_outcome'].value_counts(dropna=False)

---
## Your turn: explore before you write your proposal

Use the cells below on **your team's dataset**. Replace `df` with `trees`, `arrests`, `force`, or `stops`.

- `df.describe()` summarizes the numeric columns.
- `df['column'].value_counts(dropna=False)` counts the values in one column, including blanks.
- Make a few plots of the columns you care about.

As you explore, keep a list of everything that will need cleaning. Your proposal must name at least one cleaning step, and the six criteria in the project guide will help you shape your question.

In [ ]:
# Your code here

In [ ]:
# Your code here